# Gnomonic Expansion and Binomial Matrices
Peter Luschny, August 2026

### A first look

A 'gnomonic expansion' takes a stream of numbers and turns it into a growing rectangular table — a bit like building Pascal's Triangle row by row, except here each new number wraps an L-shaped border (a "gnomon", the ancient Greek term for the piece you add to a square to make it one size bigger) around the existing table, adding both a new row and a new column.

One side of that new border (a new row) is built by *adding up* the recent values (running sums), and the other side (a new column) is built by *subtracting* them (running differences). So a single flat sequence unfolds into a square matrix where sums and differences of the original numbers sit next to each other.

Illustrating the algorithm: Assume a 0-based sequence a = [1, 1, 2, 5, ...] and the first three steps already finished that led to the 3×3 matrix. Now, compute a(3) = 5 and place it at the lower end of the diagonal. Add a new row starting from there, adding the entry on the left in the row above and repeat this until reaching the first column: 5 + 2 -> 7 + 3 -> 10 + 5 -> 15. Next, add a new column, starting again at the new diagonal term, and subtract the term in the previous row to its left; repeat this up to the first row: 5 - 2 -> 3 - 1 -> 2 - 1 = 1.

            1   0   1  |  1      
            2   1   1  |  2      
            5   3   2  |  3      
            - - - - - - - -      
           15  10   7  |  5      

### The construction: the gnomonic expansion

Let $(a_n)_{n\ge 0}$ be a sequence of numbers. We define an increasing
sequence of square matrices $A^{(0)}\subset A^{(1)}\subset\cdots$, where
$A^{(n)}$ has size $n{+}1$, as follows.

Start with $A^{(0)} = (a_0)$. Given $A^{(n)}$, form $A^{(n+1)}$ by adding
one new row and one new column, both indexed $0,\dots,n$, sharing the value
$a_n$ at their common corner:


* New row, fill right to left: set $r_n=a_n$, and for
$k=n-1,\dots,0$,
$$
r_k = A^{(n)}_{n-1,\,k} + r_{k+1}.
$$
* New column, fill bottom to top: set $c_n = a_n$, and for
$i=n-1,\dots,0$,
$$
c_i = c_{i+1} - A^{(n)}_{i,\,n-1}.
$$

The new row becomes row $n$ of $A^{(n+1)}$, the new column becomes column
$n$, and all previously computed entries are left unchanged. We call this
process the *gnomonic expansion* of the sequence $a$.

### The binomial matrix

We associate a matrix $A(n, k), \, (n \ge 0, k \ge 0), $ with a 0-based sequence $ a(0), a(1), \ldots ,$ defined by 
$$ A(n, k) = \sum_{m=0}^d \binom{d}{m} \, s^{d - m}  a(p + m),$$
where $d = |n - k|$, $p = \min(n, k)$, and $s = 1$ if $k \le n$, otherwise $-1$. 

We call this matrix the *binomial matrix* of $a$. We also refer to the mapping $ a \mapsto A $ itself as the gnomonic expansion of $a$. The name describes the algorithmic, structural growth: each time a new term is appended to $a$, a new row is added below and a new column to the right of the existing matrix, transforming an $ n \times n $ matrix into an $ (n+1) \times (n+1) $ matrix. In this way, the given sequence $a$ is embedded as the main diagonal in the matrix. 

Further, for all $k\ge 0$,
$$
A(k,0) = \sum_{m=0}^{k}\binom{k}{m} a_m
\qquad\text{and}\qquad
A(0,k) = \sum_{m=0}^{k}\binom{k}{m}(-1)^{k-m} a_m .
$$

That is, the *first column is the binomial transform* of $ a$, and
the *first row the inverse binomial transform* of $ a$ (equivalently, the top row is the
sequence of iterated forward differences $\Delta^k a_0$).
This follows by induction: the recursive addition rule for the row is exactly Pascal's
rule $ \binom{d}{m}=\binom{d-1}{m-1}+\binom{d-1}{m} $ applied without sign,
while the subtraction rule for the column applies the same rule with
alternating sign.

### The operator view

Using the shift operator $E$ defined by $E a_n = a_{n+1}$, and the identity operator $I$ defined by $I a_n = a_n$, we can write the binomial matrix as

$$
A(n, k) =
\begin{cases}
(E+I)^{\,n-k}\,a_k, & k\le n,\\[2pt]
(E-I)^{\,k-n}\,a_n, & k> n.
\end{cases}
$$
This is just the usual binomial theorem (or the binomial expansion) applied to the operator $E+I$ or $E-I$, respectively. 

Given its particular arithmetic and algorithmic simplicity, the gnomonic expansion of a sequence is one of the fundamental methods for investigating the structure of a sequence of numbers. Combined with the concept of iterators, as built into computer languages ​​such as Python, it can be implemented very efficiently, as we show below.

### OEIS

A399000 is the main reference with an *alternative* implementation, also for Maple and Mathematica.

* A398987 Lucas numbers
* A398988 Powers n^n
* A398989 Lah numbers (sets of lists) 
* A398990 Partition numbers 
* A398991 Fubini numbers 
* A398992 Central binomial coefficients
* A398993 Bell numbers
* A398994 Schröder (big) numbers
* A398995 Pell numbers
* A398996 Motzkin numbers
* A398997 Number of involutions
* A398998 Euler numbers
* A398999 Factorial numbers
* A399000 Catalan numbers
* A399001 Fibonacci numbers
and
* A398133/A398134 Bernoulli numbers

## A Python class for gnomonic expansion

In [ ]:
from collections.abc import Iterator
from fractions import Fraction as frac
type Seq = list[int | frac]
type Matrix = list[Seq]
type SeqIterator = Iterator[int | frac]

class GnomonicExpansion:
    def __init__(self, seq: SeqIterator, dim: int | None = None) -> None:
        self.seq_source = seq
        self.matrix: Matrix = []
        if dim is not None and dim > 0:
            self.grow(dim)
        else:
            # dim = None: In this case the matrix grows until the given iterator 
            # stops (seq.next() raises StopIteration). The constant 99 is merely
            # a safety measure and was chosen arbitrarily.
            self.grow(99) 

    def grow(self, steps: int = 1) -> None:

        for _ in range(steps):
            try:
                r = next(self.seq_source)
            except StopIteration:
                break

            n = len(self.matrix)

            if n == 0:
                self.matrix.append([r])
                continue

            # Build the new lower row from the previous lower row slice and r.
            lower_new = self.matrix[-1][:] + [r]
            for k in range(n, 0, -1):
                lower_new[k - 1] += lower_new[k]

            # Build the new upper column from the previous upper column slice and r.
            prev_upper = [self.matrix[i][n - 1] for i in range(n)]
            upper_new = [0] * n + [r]
            for i in range(n - 1, -1, -1):
                upper_new[i] = upper_new[i + 1] - prev_upper[i]

            # Append the new upper values to existing rows.
            for i in range(n):
                self.matrix[i].append(upper_new[i])

            # Append the new lower row.
            self.matrix.append(lower_new)


    def get_matrix(self) -> Matrix:
        """Returns the current state of the binomial matrix."""
        return self.matrix


    def print_matrix(self) -> None:
        M = self.matrix
        n = len(M)
        
        # Width of each column
        widths = [ max(len(str(row[col])) for row in M)
                   for col in range(len(M[0])) ]

        # Markdown alignment/header row
        # c = [' ', '-', ':', ':-', '::', '::-',':::', ':::-', '::::', '::::-']
        h = ("| " + " | ".join(":-:".center(width) for width in widths) + " |")
        print(h); print(h)

        # Matrix rows
        for row in M:
            print("| " + " | ".join(f"{value:>{width}}"
                  for value, width in zip(row, widths)) + " |" )

# Catalan numbers

In [ ]:
def catalan_iterator() -> SeqIterator:
    c, n = 1, 0
    while True:
        yield c
        c = c * (4 * n + 2) // (n + 2)
        n += 1

In [ ]:
cat = GnomonicExpansion(catalan_iterator(), 8)
cat.print_matrix()

| :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-: | :-: |
| :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-: | :-: |
|    1 |    0 |    1 |    1 |   3 |   6 |  15 |  36 |
|    2 |    1 |    1 |    2 |   4 |   9 |  21 |  51 |
|    5 |    3 |    2 |    3 |   6 |  13 |  30 |  72 |
|   15 |   10 |    7 |    5 |   9 |  19 |  43 | 102 |
|   51 |   36 |   26 |   19 |  14 |  28 |  62 | 145 |
|  188 |  137 |  101 |   75 |  56 |  42 |  90 | 207 |
|  731 |  543 |  406 |  305 | 230 | 174 | 132 | 297 |
| 2950 | 2219 | 1676 | 1270 | 965 | 735 | 561 | 429 |

## Central Binomial
A000984, A002426, A026375, A163844, A163774

In [ ]:
def central_binomial_iterator() -> SeqIterator:
    b, n = 1, 0
    while True:
        yield b
        b = b * (4 * n + 2) // (n + 1)
        n += 1

In [ ]:
cbi = GnomonicExpansion(central_binomial_iterator(), 8)
cbi.print_matrix()

|  :-:  |  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|  :-:  |  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|     1 |     1 |     3 |    7 |   19 |   51 |  141 |  393 |
|     3 |     2 |     4 |   10 |   26 |   70 |  192 |  534 |
|    11 |     8 |     6 |   14 |   36 |   96 |  262 |  726 |
|    45 |    34 |    26 |   20 |   50 |  132 |  358 |  988 |
|   195 |   150 |   116 |   90 |   70 |  182 |  490 | 1346 |
|   873 |   678 |   528 |  412 |  322 |  252 |  672 | 1836 |
|  3989 |  3116 |  2438 | 1910 | 1498 | 1176 |  924 | 2508 |
| 18483 | 14494 | 11378 | 8940 | 7030 | 5532 | 4356 | 3432 |

# Fubini numbers
A000670, A052841, A000629, A089677

In [ ]:
def fubini_iterator() -> SeqIterator:
    row = [1]
    total, m = 1, 0

    while True:
        yield total
        m += 1  
        row.append(0)  
        total = 0
        for k in range(m, 0, -1):
            val = k * (row[k - 1] + row[k])
            row[k] = val
            total += val
        row[0] = 0

In [ ]:
fub = GnomonicExpansion(fubini_iterator(), 8)
fub.print_matrix()

|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|     1 |     0 |     2 |     6 |    38 |   270 |  2342 | 23646 |
|     2 |     1 |     2 |     8 |    44 |   308 |  2612 | 25988 |
|     6 |     4 |     3 |    10 |    52 |   352 |  2920 | 28600 |
|    26 |    20 |    16 |    13 |    62 |   404 |  3272 | 31520 |
|   150 |   124 |   104 |    88 |    75 |   466 |  3676 | 34792 |
|  1082 |   932 |   808 |   704 |   616 |   541 |  4142 | 38468 |
|  9366 |  8284 |  7352 |  6544 |  5840 |  5224 |  4683 | 42610 |
| 94586 | 85220 | 76936 | 69584 | 63040 | 57200 | 51976 | 47293 |

# Fibonacci numbers

In [ ]:
def fibonacci_iterator() -> SeqIterator:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

In [ ]:
fib = GnomonicExpansion(fibonacci_iterator(), 8)
fib.print_matrix()

| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|   0 |   1 |  -1 |  2 | -3 |  5 | -8 | 13 |
|   1 |   1 |   0 |  1 | -1 |  2 | -3 |  5 |
|   3 |   2 |   1 |  1 |  0 |  1 | -1 |  2 |
|   8 |   5 |   3 |  2 |  1 |  1 |  0 |  1 |
|  21 |  13 |   8 |  5 |  3 |  2 |  1 |  1 |
|  55 |  34 |  21 | 13 |  8 |  5 |  3 |  2 |
| 144 |  89 |  55 | 34 | 21 | 13 |  8 |  5 |
| 377 | 233 | 144 | 89 | 55 | 34 | 21 | 13 |

# Lucas numbers

In [ ]:
def lucas_iterator() -> SeqIterator:
    a, b = 2, 1
    while True:
        yield a
        a, b = b, a + b

In [ ]:
luc = GnomonicExpansion(lucas_iterator(), 8)
luc.print_matrix()

| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|   2 |  -1 |   3 |  -4 |   7 | -11 | 18 | -29 |
|   3 |   1 |   2 |  -1 |   3 |  -4 |  7 | -11 |
|   7 |   4 |   3 |   1 |   2 |  -1 |  3 |  -4 |
|  18 |  11 |   7 |   4 |   3 |   1 |  2 |  -1 |
|  47 |  29 |  18 |  11 |   7 |   4 |  3 |   1 |
| 123 |  76 |  47 |  29 |  18 |  11 |  7 |   4 |
| 322 | 199 | 123 |  76 |  47 |  29 | 18 |  11 |
| 843 | 521 | 322 | 199 | 123 |  76 | 47 |  29 |

# Euler numbers

In [ ]:
from math import comb

def euler_iterator() -> SeqIterator:
    seq = [1]
    yield 1
    n = 1
    sign = -1
    while True:
        yield 0  # yield 0 for odd indices 
        val = sum(comb(2 * n, 2 * k) * seq[n - k] * (1 if k % 2 == 1 else -1)
              for k in range(1, n + 1))
        seq.append(val)
        yield sign * val
        sign = -sign
        n += 1

In [ ]:
eul = GnomonicExpansion(euler_iterator(), 9)
eul.print_matrix()

| :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
| :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|    1 |   -1 |    0 |    2 |    0 |  -16 |    0 |  272 |    0 |
|    1 |    0 |   -1 |    2 |    2 |  -16 |  -16 |  272 |  272 |
|    0 |   -1 |   -1 |    1 |    4 |  -14 |  -32 |  256 |  544 |
|   -2 |   -2 |   -1 |    0 |    5 |  -10 |  -46 |  224 |  800 |
|    0 |    2 |    4 |    5 |    5 |   -5 |  -56 |  178 | 1024 |
|   16 |   16 |   14 |   10 |    5 |    0 |  -61 |  122 | 1202 |
|    0 |  -16 |  -32 |  -46 |  -56 |  -61 |  -61 |   61 | 1324 |
| -272 | -272 | -256 | -224 | -178 | -122 |  -61 |    0 | 1385 |
|    0 |  272 |  544 |  800 | 1024 | 1202 | 1324 | 1385 | 1385 |

# Pell numbers

A000129, A016116, A007052, [A077957, A007070]

In mathematics, the Pell numbers are an infinite sequence of integers, known since ancient times, that comprise the denominators of the closest rational approximations to the square root of 2. This sequence of approximations begins ⁠
1/1⁠, ⁠3/2⁠, ⁠7/5⁠, ⁠17/12⁠, and ⁠41/29⁠, so the sequence of Pell numbers begins with 1, 2, 5, 12, and 29 (from Wikipedia).

In [ ]:
def pell_iterator() -> SeqIterator:
    a, b = 0, 1
    while True:
        yield b
        a, b = b, a + 2 * b

In [ ]:
pel = GnomonicExpansion(pell_iterator(), 9)
pel.print_matrix()

|  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-: |
|  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-: |
|     1 |     1 |    2 |    2 |    4 |    4 |    8 |    8 |  16 |
|     3 |     2 |    3 |    4 |    6 |    8 |   12 |   16 |  24 |
|    10 |     7 |    5 |    7 |   10 |   14 |   20 |   28 |  40 |
|    34 |    24 |   17 |   12 |   17 |   24 |   34 |   48 |  68 |
|   116 |    82 |   58 |   41 |   29 |   41 |   58 |   82 | 116 |
|   396 |   280 |  198 |  140 |   99 |   70 |   99 |  140 | 198 |
|  1352 |   956 |  676 |  478 |  338 |  239 |  169 |  239 | 338 |
|  4616 |  3264 | 2308 | 1632 | 1154 |  816 |  577 |  408 | 577 |
| 15760 | 11144 | 7880 | 5572 | 3940 | 2786 | 1970 | 1393 | 985 |

# Riordan numbers
| A005043 | A126930 | A000108 | A106640 |

In [ ]:
def riordan_iterator() -> SeqIterator:
    b, a, n, r = 1, 0, 1, 0
    yield 1
    while True:
        yield r
        r = n * (2 * a + 3 * b) // (n + 2)
        b, a, n = a, r, n + 1

In [ ]:
rio = GnomonicExpansion(riordan_iterator(), 9)
rio.print_matrix()

| :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|    1 |   -1 |   2 |  -3 |   6 | -10 |  20 | -35 | 70 |
|    1 |    0 |   1 |  -1 |   3 |  -4 |  10 | -15 | 35 |
|    2 |    1 |   1 |   0 |   2 |  -1 |   6 |  -5 | 20 |
|    5 |    3 |   2 |   1 |   2 |   1 |   5 |   1 | 15 |
|   14 |    9 |   6 |   4 |   3 |   3 |   6 |   6 | 16 |
|   42 |   28 |  19 |  13 |   9 |   6 |   9 |  12 | 22 |
|  132 |   90 |  62 |  43 |  30 |  21 |  15 |  21 | 34 |
|  429 |  297 | 207 | 145 | 102 |  72 |  51 |  36 | 55 |
| 1430 | 1001 | 704 | 497 | 352 | 250 | 178 | 127 | 91 |

# Motzkin numbers

In [ ]:
def motzkin_iterator() -> SeqIterator:
    a, b = 1, 1
    yield a
    yield b
    n = 2
    while True:
        m = ((2 * n + 1) * b + (3 * n - 3) * a) // (n + 2)
        yield m
        a, b, n = b, m, n + 1

In [ ]:
mot = GnomonicExpansion(motzkin_iterator(), 9)
mot.print_matrix()

| :-:  | :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-: | :-: |
| :-:  | :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-: | :-: |
|    1 |    0 |    1 |    0 |    2 |   0 |   5 |   0 |  14 |
|    2 |    1 |    1 |    1 |    2 |   2 |   5 |   5 |  14 |
|    5 |    3 |    2 |    2 |    3 |   4 |   7 |  10 |  19 |
|   14 |    9 |    6 |    4 |    5 |   7 |  11 |  17 |  29 |
|   42 |   28 |   19 |   13 |    9 |  12 |  18 |  28 |  46 |
|  132 |   90 |   62 |   43 |   30 |  21 |  30 |  46 |  74 |
|  429 |  297 |  207 |  145 |  102 |  72 |  51 |  76 | 120 |
| 1430 | 1001 |  704 |  497 |  352 | 250 | 178 | 127 | 196 |
| 4862 | 3432 | 2431 | 1727 | 1230 | 878 | 628 | 450 | 323 |

# Schröder numbers
A006318, A174347, A052709

In [ ]:
def schroeder_iterator() -> SeqIterator:
    b, a, n = 1, 2, 3
    yield b
    yield a

    while True:
        t = a * (6 * n - 9) - (n - 3) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

In [ ]:
sch = GnomonicExpansion(schroeder_iterator(), 8)
sch.print_matrix()

|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  | :-:  |
|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  | :-:  |
|     1 |     1 |     3 |     9 |    31 |   113 |   431 | 1697 |
|     3 |     2 |     4 |    12 |    40 |   144 |   544 | 2128 |
|    11 |     8 |     6 |    16 |    52 |   184 |   688 | 2672 |
|    47 |    36 |    28 |    22 |    68 |   236 |   872 | 3360 |
|   223 |   176 |   140 |   112 |    90 |   304 |  1108 | 4232 |
|  1135 |   912 |   736 |   596 |   484 |   394 |  1412 | 5340 |
|  6063 |  4928 |  4016 |  3280 |  2684 |  2200 |  1806 | 6752 |
| 33535 | 27472 | 22544 | 18528 | 15248 | 12564 | 10364 | 8558 |

# Jacobsthal
A001045, A000244, A020988

In [ ]:
def jacobsthal_iterator() -> SeqIterator:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, b + 2 * a

In [ ]:
jac = GnomonicExpansion(jacobsthal_iterator(), 9)
jac.print_matrix()

| :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|    0 |    1 |  -1 |   3 |  -5 |  11 | -21 |  43 | -85 |
|    1 |    1 |   0 |   2 |  -2 |   6 | -10 |  22 | -42 |
|    3 |    2 |   1 |   2 |   0 |   4 |  -4 |  12 | -20 |
|    9 |    6 |   4 |   3 |   2 |   4 |   0 |   8 |  -8 |
|   27 |   18 |  12 |   8 |   5 |   6 |   4 |   8 |   0 |
|   81 |   54 |  36 |  24 |  16 |  11 |  10 |  12 |   8 |
|  243 |  162 | 108 |  72 |  48 |  32 |  21 |  22 |  20 |
|  729 |  486 | 324 | 216 | 144 |  96 |  64 |  43 |  42 |
| 2187 | 1458 | 972 | 648 | 432 | 288 | 192 | 128 |  85 |

# Central Delannoy
| A001850 |  A080609 | A006139 |

In [ ]:
def delannoy_iterator() -> SeqIterator:
    b, a, n = 1, 3, 2
    yield b
    yield a

    while True:
        t = a * (6 * n - 3) - (n - 1) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

In [ ]:
dela = GnomonicExpansion(delannoy_iterator(), 8)
dela.print_matrix()

|  :-:   |  :-:   |  :-:   |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|  :-:   |  :-:   |  :-:   |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|      1 |      2 |      8 |    32 |   136 |   592 |  2624 | 11776 |
|      4 |      3 |     10 |    40 |   168 |   728 |  3216 | 14400 |
|     20 |     16 |     13 |    50 |   208 |   896 |  3944 | 17616 |
|    112 |     92 |     76 |    63 |   258 |  1104 |  4840 | 21560 |
|    664 |    552 |    460 |   384 |   321 |  1362 |  5944 | 26400 |
|   4064 |   3400 |   2848 |  2388 |  2004 |  1683 |  7306 | 32344 |
|  25376 |  21312 |  17912 | 15064 | 12676 | 10672 |  8989 | 39650 |
| 160640 | 135264 | 113952 | 96040 | 80976 | 68300 | 57628 | 48639 |

# Bell numbers

In [ ]:
from itertools import accumulate

def bell_iterator() -> SeqIterator:
    row = [1]
    while True:
        yield row[0]
        row = list(accumulate([row[-1], *row]))

In [ ]:
bell = GnomonicExpansion(bell_iterator(), 9)
bell.print_matrix()

|  :-:  |  :-:  |  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|  :-:  |  :-:  |  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|     1 |     0 |     1 |     1 |    4 |   11 |   41 |  162 |  715 |
|     2 |     1 |     1 |     2 |    5 |   15 |   52 |  203 |  877 |
|     5 |     3 |     2 |     3 |    7 |   20 |   67 |  255 | 1080 |
|    15 |    10 |     7 |     5 |   10 |   27 |   87 |  322 | 1335 |
|    52 |    37 |    27 |    20 |   15 |   37 |  114 |  409 | 1657 |
|   203 |   151 |   114 |    87 |   67 |   52 |  151 |  523 | 2066 |
|   877 |   674 |   523 |   409 |  322 |  255 |  203 |  674 | 2589 |
|  4140 |  3263 |  2589 |  2066 | 1657 | 1335 | 1080 |  877 | 3263 |
| 21147 | 17007 | 13744 | 11155 | 9089 | 7432 | 6097 | 5017 | 4140 |

# Pólya Trees
A000081, A034781, A375467

In [ ]:
from math import isqrt

def polyatree_iterator()  -> SeqIterator:
    yield 0
    yield 1
    t = [0, 1]; t_append = t.append
    D = [0, 1]; D_append = D.append
    i = 2

    while True:
        total = sum(t[i - j] * D[j] for j in range(1, i))
        t_i = total // (i - 1)
        t_append(t_i)
        yield t_i

        d_i = 0
        for d in range(1, isqrt(i) + 1):
            if i % d == 0:
                d_i += d * t[d]
                if d * d != i:
                    d2 = i // d
                    d_i += d2 * t[d2]
        D_append(d_i)
        i += 1

In [ ]:
pti = GnomonicExpansion(polyatree_iterator(), 9)
pti.print_matrix()

| :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|    0 |    1 |  -1 |   2 |  -2 |   4 |  -5 |  13 | -25 |
|    1 |    1 |   0 |   1 |   0 |   2 |  -1 |   8 | -12 |
|    3 |    2 |   1 |   1 |   1 |   2 |   1 |   7 |  -4 |
|    8 |    5 |   3 |   2 |   2 |   3 |   3 |   8 |   3 |
|   22 |   14 |   9 |   6 |   4 |   5 |   6 |  11 |  11 |
|   64 |   42 |  28 |  19 |  13 |   9 |  11 |  17 |  22 |
|  195 |  131 |  89 |  61 |  42 |  29 |  20 |  28 |  39 |
|  615 |  420 | 289 | 200 | 139 |  97 |  68 |  48 |  67 |
| 1991 | 1376 | 956 | 667 | 467 | 328 | 231 | 163 | 115 |

# Sets of lists   [Lah]

In [ ]:
def lah_iterator() -> SeqIterator:
    b, a, n = 1, 1, 2
    yield b
    yield a

    while True:
        q = (2 * n - 1) * a - (n - 1) * (n - 2) * b
        b, a, n = a, q, n + 1
        yield q

In [ ]:
lah = GnomonicExpansion(lah_iterator(), 8)
lah.print_matrix()

|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|     1 |     0 |     2 |     6 |    36 |   240 |  1920 | 17640 |
|     2 |     1 |     2 |     8 |    42 |   276 |  2160 | 19560 |
|     6 |     4 |     3 |    10 |    50 |   318 |  2436 | 21720 |
|    26 |    20 |    16 |    13 |    60 |   368 |  2754 | 24156 |
|   148 |   122 |   102 |    86 |    73 |   428 |  3122 | 26910 |
|  1032 |   884 |   762 |   660 |   574 |   501 |  3550 | 30032 |
|  8464 |  7432 |  6548 |  5786 |  5126 |  4552 |  4051 | 33582 |
| 79592 | 71128 | 63696 | 57148 | 51362 | 46236 | 41684 | 37633 |

# Involutions (Young tableaux with n cells)

A000085, A005425, A123023, A378100

In [ ]:
def involution_iterator() -> SeqIterator:
    a, b, n = 1, 1, 1
    yield a
    yield b

    while True:
        a, b = b, b + n * a
        n += 1
        yield b

In [ ]:
inv = GnomonicExpansion(involution_iterator(), 9)
inv.print_matrix()

| :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-: | :-: |
| :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-:  | :-: | :-: |
|    1 |    0 |    1 |    0 |    3 |    0 |   15 |   0 | 105 |
|    2 |    1 |    1 |    1 |    3 |    3 |   15 |  15 | 105 |
|    5 |    3 |    2 |    2 |    4 |    6 |   18 |  30 | 120 |
|   14 |    9 |    6 |    4 |    6 |   10 |   24 |  48 | 150 |
|   43 |   29 |   20 |   14 |   10 |   16 |   34 |  72 | 198 |
|  142 |   99 |   70 |   50 |   36 |   26 |   50 | 106 | 270 |
|  499 |  357 |  258 |  188 |  138 |  102 |   76 | 156 | 376 |
| 1850 | 1351 |  994 |  736 |  548 |  410 |  308 | 232 | 532 |
| 7193 | 5343 | 3992 | 2998 | 2262 | 1714 | 1304 | 996 | 764 |

# Factorial numbers
A000142, A000166, A000522, A002627, A002467.

In [ ]:
def factorial_iterator() -> SeqIterator:
    a, n = 1, 1
    while True:
        yield a
        a, n = a * n, n + 1

In [ ]:
fac = GnomonicExpansion(factorial_iterator(), 8)
fac.print_matrix()

|  :-:  |  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|  :-:  |  :-:  |  :-:  | :-:  | :-:  | :-:  | :-:  | :-:  |
|     1 |     0 |     1 |    2 |    9 |   44 |  265 | 1854 |
|     2 |     1 |     1 |    3 |   11 |   53 |  309 | 2119 |
|     5 |     3 |     2 |    4 |   14 |   64 |  362 | 2428 |
|    16 |    11 |     8 |    6 |   18 |   78 |  426 | 2790 |
|    65 |    49 |    38 |   30 |   24 |   96 |  504 | 3216 |
|   326 |   261 |   212 |  174 |  144 |  120 |  600 | 3720 |
|  1957 |  1631 |  1370 | 1158 |  984 |  840 |  720 | 4320 |
| 13700 | 11743 | 10112 | 8742 | 7584 | 6600 | 5760 | 5040 |

# Double factorial numbers
A006882, A263529, A262020

In [ ]:
def doublefactorial_iterator() -> SeqIterator:
    a, b, n = 1, 1, 1
    while True:
        yield b
        a, b, n = b, a * n, n + 1

In [ ]:
dfac = GnomonicExpansion(doublefactorial_iterator(), 9)
dfac.print_matrix()

| :-:  | :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-:  | :-: |
| :-:  | :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-:  | :-: |
|    1 |    0 |    1 |   -1 |    5 | -11 |  43 | -127 | 489 |
|    2 |    1 |    1 |    0 |    4 |  -6 |  32 |  -84 | 362 |
|    5 |    3 |    2 |    1 |    4 |  -2 |  26 |  -52 | 278 |
|   13 |    8 |    5 |    3 |    5 |   2 |  24 |  -26 | 226 |
|   37 |   24 |   16 |   11 |    8 |   7 |  26 |   -2 | 200 |
|  111 |   74 |   50 |   34 |   23 |  15 |  33 |   24 | 198 |
|  355 |  244 |  170 |  120 |   86 |  63 |  48 |   57 | 222 |
| 1191 |  836 |  592 |  422 |  302 | 216 | 153 |  105 | 279 |
| 4201 | 3010 | 2174 | 1582 | 1160 | 858 | 642 |  489 | 384 |

# Number of automorphisms, n^n

In [ ]:
def self_powers_iterator() -> SeqIterator:
    n = 0
    while True:
        yield n ** n
        n += 1

In [ ]:
endo = GnomonicExpansion(self_powers_iterator(), 7)
endo.print_matrix()

|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |  :-:  |
|     1 |     0 |     3 |    17 |   169 |  2079 | 31261 |
|     2 |     1 |     3 |    20 |   186 |  2248 | 33340 |
|     7 |     5 |     4 |    23 |   206 |  2434 | 35588 |
|    43 |    36 |    31 |    27 |   229 |  2640 | 38022 |
|   393 |   350 |   314 |   283 |   256 |  2869 | 40662 |
|  4721 |  4328 |  3978 |  3664 |  3381 |  3125 | 43531 |
| 69853 | 65132 | 60804 | 56826 | 53162 | 49781 | 46656 |

## LCM -- Least common multiple of \{1, 2, ..., n\} 

In [ ]:
def LCM_iterator(lng: int) -> SeqIterator:
    if lng <= 0: 
        print("This iterator requires a positive run length.")
        return

    lambd = [1] * lng
    lcm = [1] * lng
    isp = [True] * lng

    for p in range(2, lng):
        if isp[p]:
            for i in range(p * p, lng, p):
                isp[i] = False
            k = p
            while k < lng:
                lambd[k] = p
                k *= p
        lcm[p] = lcm[p - 1] * lambd[p]

    yield from lcm

In [ ]:
gen = LCM_iterator(8)
lcm = GnomonicExpansion(gen)
lcm.print_matrix()

| :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-:  | :-:  |
| :-:  | :-:  | :-:  | :-:  | :-: | :-: | :-:  | :-:  |
|    1 |    0 |    1 |    2 |  -3 |  44 | -215 | 1014 |
|    2 |    1 |    1 |    3 |  -1 |  41 | -171 |  799 |
|    5 |    3 |    2 |    4 |   2 |  40 | -130 |  628 |
|   16 |   11 |    8 |    6 |   6 |  42 |  -90 |  498 |
|   53 |   37 |   26 |   18 |  12 |  48 |  -48 |  408 |
|  206 |  153 |  116 |   90 |  72 |  60 |    0 |  360 |
|  757 |  551 |  398 |  282 | 192 | 120 |   60 |  360 |
| 2780 | 2023 | 1472 | 1074 | 792 | 600 |  480 |  420 |

# Partition numbers (integer) 

In [ ]:
def partitions_iterator() -> SeqIterator:
    p = [1]
    yield p[0]
    n = 1
    while True:
        total = 0
        k = 1
        while True:
            # generalized pentagonal number
            g1 = k * (3 * k - 1) // 2
            if g1 > n: break
            sign = 1 if k % 2 else -1
            total += sign * p[n - g1]
            g2 = k * (3 * k + 1) // 2
            if g2 <= n:
                total += sign * p[n - g2]
            k += 1
        p.append(total)
        yield total
        n += 1

In [ ]:
pari = GnomonicExpansion(partitions_iterator(), 9)
pari.print_matrix()

| :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|    1 |   0 |   1 |  -1 |   2 |  -4 |  9 | -21 | 49 |
|    2 |   1 |   1 |   0 |   1 |  -2 |  5 | -12 | 28 |
|    5 |   3 |   2 |   1 |   1 |  -1 |  3 |  -7 | 16 |
|   13 |   8 |   5 |   3 |   2 |   0 |  2 |  -4 |  9 |
|   34 |  21 |  13 |   8 |   5 |   2 |  2 |  -2 |  5 |
|   88 |  54 |  33 |  20 |  12 |   7 |  4 |   0 |  3 |
|  225 | 137 |  83 |  50 |  30 |  18 | 11 |   4 |  3 |
|  569 | 344 | 207 | 124 |  74 |  44 | 26 |  15 |  7 |
| 1425 | 856 | 512 | 305 | 181 | 107 | 63 |  37 | 22 |

# Bernoulli Matrix

In [ ]:
def bernoulli_seidel() -> SeqIterator:
    """Generates Bernoulli numbers with B_1 = 1/2."""
    yield frac(1)     # B_0 = 1
    yield frac(1, 2)  # B_1 = 1/2

    ZERO = frac(0)  # odd-indexed Bernoulli number past B_1 vanish
    row = [1]       # B_0
    p2 = 8          # 2^(2+1) = 8
    m = 1           # row length

    while True:
        f = frac(row[-1], p2 - 2)
        yield -f if m % 2 == 0 else f  # yield B_n 
        yield ZERO  # yield B_{n+1} (always 0)
        row.append(0)
        p2 <<= 2
        m += 1
        for k in range(m - 2, -1, -1): row[k] += row[k + 1]
        for k in range(1, m): row[k] += row[k - 1]

In [ ]:
ber = GnomonicExpansion(bernoulli_seidel(), 8)
ber.print_matrix()

|  :-:   |  :-:  |  :-:   |  :-:   |  :-:   |  :-:  |  :-:   |  :-:   |
|  :-:   |  :-:  |  :-:   |  :-:   |  :-:   |  :-:  |  :-:   |  :-:   |
|      1 |  -1/2 |    1/6 |      0 |  -1/30 |     0 |   1/42 |      0 |
|    3/2 |   1/2 |   -1/3 |    1/6 |  -1/30 | -1/30 |   1/42 |   1/42 |
|   13/6 |   2/3 |    1/6 |   -1/6 |   2/15 | -1/15 | -1/105 |   1/21 |
|      3 |   5/6 |    1/6 |      0 |  -1/30 |  1/15 | -8/105 |  4/105 |
| 119/30 | 29/30 |   2/15 |  -1/30 |  -1/30 |  1/30 | -1/105 | -4/105 |
|      5 | 31/30 |   1/15 |  -1/15 |  -1/30 |     0 |   1/42 |  -1/21 |
| 253/42 | 43/42 | -1/105 | -8/105 | -1/105 |  1/42 |   1/42 |  -1/42 |
|      7 | 41/42 |  -1/21 | -4/105 |  4/105 |  1/21 |   1/42 |      0 |

# Use with an arbitrary sequence

You can wrap any list of integers with Python's 'iter' operator to use it with the Gnomonic Expansion class.

In [ ]:
numbers = [1, 2, 3, 5, 7, 11, 13, 17, 19, 23]
num_iteraor = iter(numbers)
seqit = GnomonicExpansion(num_iteraor)
seqit.print_matrix()

| :-:  | :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| :-:  | :-:  | :-:  | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
|    1 |    1 |    0 |   1 |  -2 |   5 | -14 | 37 | -90 | 205 |
|    3 |    2 |    1 |   1 |  -1 |   3 |  -9 | 23 | -53 | 115 |
|    8 |    5 |    3 |   2 |   0 |   2 |  -6 | 14 | -30 |  62 |
|   21 |   13 |    8 |   5 |   2 |   2 |  -4 |  8 | -16 |  32 |
|   54 |   33 |   20 |  12 |   7 |   4 |  -2 |  4 |  -8 |  16 |
|  137 |   83 |   50 |  30 |  18 |  11 |   2 |  2 |  -4 |   8 |
|  342 |  205 |  122 |  72 |  42 |  24 |  13 |  4 |  -2 |   4 |
|  837 |  495 |  290 | 168 |  96 |  54 |  30 | 17 |   2 |   2 |
| 2006 | 1169 |  674 | 384 | 216 | 120 |  66 | 36 |  19 |   4 |
| 4713 | 2707 | 1538 | 864 | 480 | 264 | 144 | 78 |  42 |  23 |